# **LLM + RAG System with Tabular Information**
**Natural Language Processing**  
**Master's Degree in Applied Artificial Intelligence**  
**Tecnológico de Monterrey**  

**Team Involved**
*   Mario Alberto Guillen De La Torre - A01796701
*   Miguel Angel Fernandez Castresana - A01796998
*   Gilberto Valencia Acosta - A01273602
*   Francisco Daniel Valdes Escarrega - A01796968

In [ ]:
# We import all the libraries to be used; we will use ChromaDB to create the vector space used by RAG
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from transformers import BitsAndBytesConfig

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sentence_transformers import SentenceTransformer

import glob
import os
from os import walk

import chromadb
import warnings

import pypdf

warnings.filterwarnings("ignore")

In [ ]:
# We make sure we are using our local dedicated GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: NVIDIA GeForce RTX 4070 Ti


### **Reading the data**

In [ ]:
# We save the different paths used by our notebook
BASE_DIR = r"."
DOCUMENT_DIR   = f"{BASE_DIR}/Documentos"
CHROMA_DIR   = f"{BASE_DIR}/chroma"
CHECKPOINTS_DIR = f"{BASE_DIR}/checkpoints"
TRAINING_DIR = f"{BASE_DIR}/training_data"
MODELO_FINAL_DIR = f"{BASE_DIR}/modelo_final_rag"
filenames = next(walk(DOCUMENT_DIR ), (None, None, []))[2]

In [ ]:
# If the paths don't exist, we create them
os.makedirs(CHROMA_DIR,exist_ok=True)
os.makedirs(CHECKPOINTS_DIR,exist_ok=True)
os.makedirs(MODELO_FINAL_DIR,exist_ok=True)

In [ ]:
# We save the text of each document
document_text = []
for i, docu in enumerate(filenames):
    filedocu = DOCUMENT_DIR+'/' + docu
    reader = pypdf.PdfReader(filedocu)
    if len(reader.pages) > 0:
        for n in range(len(reader.pages)):
            document_text.append(reader.pages[n].extract_text())

    print('Document '+str(i+1)+' Processing done')

Document 1 Processing done
Document 2 Processing done


In [ ]:
# We save the text of each document in chunks to be used later
def chunk_text(text,chunk_size = 50, overlap=10):
    return [text[i:i+chunk_size] for i in range(0,len(text),chunk_size-overlap)]

chunks = []
for doc in document_text:
    chunks.extend(chunk_text(doc, chunk_size = 600, overlap=150))

### **Building the RAG space**

In [ ]:
# Include your own login token
from huggingface_hub import login
loginToken = 'YOURLOGINTOKEN'
login(token=loginToken)

In [ ]:
# We save the relevant Chroma variables 
client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_or_create_collection(name="rag_demo")

In [ ]:
# We load the model that will be used to create the embedded vectors of our files' chunks
embedding_model = SentenceTransformer("intfloat/multilingual-e5-large")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
# We create the collection of embedded vectors and save it in ChromaDB
if collection.count() == 0:
    embeddings = embedding_model.encode(chunks).tolist()
    collection.add(
        documents = chunks,
        embeddings= embeddings,
        ids = [str(i) for i in range(len(chunks))]
    )

In [ ]:
# This is the function that returns the k closest chunks to the provided query to perform RAG
def retrieve(query,k=3):
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results = k
    )
    return results["documents"][0]

### **Loading the LLM and the Tokenizer**

In [ ]:
# Model chosen to answer the question asked by the user, this one is multilingual
model_name = "Qwen/Qwen2.5-3B-Instruct"

In [ ]:
# We load the model's tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# We load the model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

### **Q&A System**

In [ ]:
# Functions responsible for building the prompt and the RAG
def build_prompt(contexto,pregunta):
    txt_contexto = "\n\n".join(contexto)
    messages = [
        {
            "role": "system",
            "content": (
                "Responde usando SOLO el contexto dado. Sé breve y directo.\n"
                "Reglas estrictas:\n"
                "- Si la información no está en el contexto, responde exactamente: No lo sé.\n"
                "- Siempre respondes con la información correcta.\n"
                "- No repitas frases ni ideas.\n"
                "- No elabores más allá de lo pedido.\n"
                "- Cuando termines, deja de escribir inmediatamente."
            )
        },
        {
            "role": "user",
            "content": f"Contexto:\n{txt_contexto}\n\nPregunta: {pregunta}"
        }
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True 
    )

def rag_answer(pregunta):
    contexto = retrieve(pregunta,k=3)
    prompt = build_prompt(contexto, pregunta)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs["input_ids"].shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens = 350,
        temperature=0.9,
        top_p = 0.9,
        top_k=50,
        pad_token_id = tokenizer.eos_token_id,
        eos_token_id= tokenizer.eos_token_id,
        repetition_penalty=1.15
    )

    new_tokens = outputs[0][input_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

### **Q&A Testing**

In [19]:
print(rag_answer("Según la sección de 'Exceptions', ¿qué excepción se lanza si divido entre cero?"))

ZeroDivisionError


In [20]:
print(rag_answer("¿Puedes proporcionarme solamente 2 ejemplos de métodos de cadena (string methods)?"))

"a".upper()                     # "A"
"a".lower()                     # "a"


In [21]:
print(rag_answer("¿Cuáles son las principales desventajas de utilizar el modelo Bosque Aleatorio (Random Forests)?"))

Sensitivo a los outliers y puede causar un sobreajuste.
Con alta complejidad computacional.


In [22]:
print(rag_answer("¿Cuáles son 3 casos de uso (use cases) de las técnicas de aprendizaje no supervisado (Unsupervised Learning) para las técnicas de agrupamiento (clustering)?"))

1. Detección de fraude
2. Análisis de segmentación de clientes
3. Identificación de patrones en datos grandes


In [23]:
print(rag_answer("¿Cuáles las funciones para convertir el tipo de dato en python?"))

abs(), round(), min(), max(), sum()


In [32]:
print(rag_answer("¿Cuáles es la descripcion de regresion linear?"))

Un algoritmo que modela una relación lineal entre entradas y un valor numérico continuo de salida. Es menos susceptible a los outliers y puede subajustarse con datos grandes y altamente dimensionales.


### **Saving the model**

In [27]:

tokenizer.save_pretrained(MODELO_FINAL_DIR)

('./modelo_final_rag\\tokenizer_config.json',
 './modelo_final_rag\\chat_template.jinja',
 './modelo_final_rag\\tokenizer.json')

### **Final Thoughts:**

This technique is especially useful for organizations that store large volumes of information in documents, such as regulations, product manuals, procedures, or internal documentation, since it allows that information to be queried efficiently using natural language.
However, to achieve satisfactory performance, it was necessary to adjust several system parameters:

- The chunk size and the overlap between chunks were increased considerably, since with smaller values the model rarely had enough context to correctly answer complex questions.
- Larger language and embedding models with multilingual support were selected. This made it possible to make queries in Spanish even when the documents were in English.
- Quantization and fine-tuning were not necessary for this implementation. In the tests performed, both techniques produced lower performance than the base model, which is probably due to the small size of the dataset used.